In [1]:
import mesa
from mesa.discrete_space import FixedAgent, OrthogonalMooreGrid
import pandas as pd
import numpy as np
from scipy.stats import bernoulli
import matplotlib.pyplot as plt
import seaborn as sns
import cognitive_functions as cf

In [33]:
class individual(FixedAgent):
    """An agent that chooses between two options based on expected utility."""
    def __init__(self, model, lambd, gamma, reward_rb, reward_rf, cost_rb, min_start_wealth, p, beta, alpha, num_neighbors=20):
        """ Create a new decision maker agent.
        Args:
            self: Object that stores characteristics of the agent.
            model: Reference to the model this agent belongs to.
            num_neighbors: Number of neighbors to consider for relative desperation.
            gamma: Utility parameter.
            reward_rb: Reward for rule breaking.
            reward_rf: Reward for following rules.
            cost_rb: Cost of rule breaking.
            min_start_wealth: The  minimum starting wealth of the agent.
            p: Probability of being caught for rule-breaking.
            beta: Optimism/pessimism.
            alpha: Likelihood sensitivity.
            desperate_state: State of relative desperation (0 = not desperate, 1 = desperate).
            radius: Radius for neighborhood search.
        """
        super().__init__(model)
        self.lambd = lambd
        self.gamma = gamma
        self.reward_rb = reward_rb
        self.reward_rf = reward_rf
        self.cost_rb = cost_rb
        self.p = p
        self.beta = beta
        self.alpha = alpha

        # self.start_wealth = (np.random.pareto(a=1.7, size=1)+1) * min_start_wealth
        # self.wealth = self.start_wealth
        # self.end_wealth = self.wealth 

        self.wealth = np.random.pareto(a=1.7, size=1)[0] * min_start_wealth + 1  # Ensure wealth is above the minimum

        self.num_neighbors = num_neighbors
        self.desperate_state = 0  # Initialize desperate state to 0 (not desperate)
        self.decision = 0  # Initialize decision to 0 (follow rules)
        self.caught = False  # Initialize caught state to False
        self.income_rank = 0  # Initialize income rank to 0
        self.rb_choice = 0  # Initialize rule-breaking choice to 0 (follow rules)

    def relative_desperation(self):
        """Determine the relative desperation of the agent based on income rank."""
        # Get neighbors within the specified radius
        neighbors = self.model.grid.get_neighbors(self.pos, moore=True, include_center=False, radius=20)

        # Get a vector of the wealth of the agents in the neighborhood
        neighbor_wealths = [neighbor.wealth for neighbor in neighbors] 
        # Count the number of neighbors with wealth lower than the agent's wealth
        num_poorer_neighbors = sum(1 for w in neighbor_wealths if w <= self.wealth)

        # Calculate the income rank of the agent in its neighborhood
        income_rank = cf.income_rank(i=num_poorer_neighbors, n=len(neighbors))
        self.income_rank = income_rank  # Store the income rank in the agent's attributes

        # Label the agent as desperate if their income rank is less than 0
        if income_rank < 0:
            self.desperate_state = 1  # Agent is relatively desperate
            print(f"Agent {self.unique_id} is relatively desperate with income rank {income_rank}")
        else:
            self.desperate_state = 0  # Agent is not relatively desperate
            print(f"Agent {self.unique_id} is not relatively desperate with income rank {income_rank}")


    def compute_expected_utilities(self):
        """Choose an option based on subjective value. If agent is relatively desperate, use a different utility function."""
        if self.desperate_state == 1:
            # If the agent is relatively desperate, use a different utility function
            self.SV_rule_break = cf.SV_relative_desp_RB(
            gamma=self.gamma, starting_wealth=self.wealth, lambd=self.lambd, p=self.p, beta=self.beta, alpha=self.alpha, reward_rb=self.reward_rb, cost_rb=self.cost_rb)
        else:
            self.SV_rule_break = cf.SV_rule_break(gamma=self.gamma, reward_rb=self.reward_rb, cost_rb=self.cost_rb, p=self.p, beta=self.beta, alpha=self.alpha, starting_wealth=self.wealth)  # SV of rule breaking
        self.SV_follow_rules = cf.SV_follow_rules(reward_rf=self.reward_rf, starting_wealth=self.wealth, gamma=self.gamma) # SV of following rules
        

        # Compute the softmax probabilities
        probs = cf.softmax(self.SV_rule_break, self.SV_follow_rules)
        
        rb_choice = bernoulli.rvs(probs[0])  # Bernoulli trial for rule breaking choice
        if rb_choice==1:
            self.decision = 1 # breaks rules
            print(f"Agent {self.unique_id} chose to break the rules with probability {probs[0]}")
        else:
            self.decision = 0 # follows rules
            print(f"Agent {self.unique_id} chose to follow the rules with probability {probs[1]}")
        return rb_choice
    
    def update_wealth(self):
        """Update the agent's wealth based on the choice made."""
        self.start_wealth = self.wealth
        print(f"Agent {self.unique_id} current wealth before decision: {self.start_wealth}")
        rb_choice = self.compute_expected_utilities()  # Get the choice made by the agent
        if rb_choice==1:
            # If the agent chooses to break the rules and is NOT caught
            if bernoulli.rvs(1 - self.p) == 1:
                self.caught = False
                self.wealth += self.reward_rb
                print(f"Agent {self.unique_id} broke the rules successfully and gained {self.reward_rb}. Current wealth: {self.end_wealth}")
            # If the agent chooses to break the rules and is caught
            else:
                self.caught = True
                self.wealth -= self.cost_rb
                print(f"Agent {self.unique_id} was caught breaking the rules and lost {self.cost_rb}. Current wealth: {self.end_wealth}")
        else:
            # If the agent chooses to follow the rules
            self.wealth += self.reward_rf
        self.wealth -= 100 # Deduct the cost of living (e.g., paying for food, rent, etc.)
        self.end_wealth = self.wealth
        print(f"Agent {self.unique_id} updated wealth: {self.end_wealth}")

    
    def step(self):
        self.wealth_start = self.wealth
        self.relative_desperation() # Determine if the agent is relatively desperate
        self.compute_expected_utilities()

        # Update wealth based on decision
        if self.decision == 1: # breaks rules
            # If the agent chooses to break the rules and is NOT caught
            if bernoulli.rvs(1 - self.p) == 1:
                self.caught = False
                self.wealth += self.reward_rb
            # If the agent chooses to break the rules and is caught
            else:
                self.caught = True
                self.wealth -= self.cost_rb
        else: 
            # If the agent chooses to follow the rules
            self.wealth += self.reward_rf
        # Deduct the cost of living (e.g., paying for food, rent, etc.)
        self.wealth -= 100

        # self.update_wealth() # Update the agent's wealth based on the choice made
        self.wealth_end = self.wealth


In [34]:
def crime_proportion(model):
    agent_crimes = [agent.decision for agent in model.agents]
    n = model.width * model.height  # Total number of agents in the grid
    proportion = (sum(agent_crimes) ) / n
    return proportion


In [35]:
from mesa import Model
from mesa.datacollection import DataCollector
from mesa.space import SingleGrid
import numpy as np

# from .agents import decision_maker

class rel_DMAP_model(Model):
    """A model with a number of decision makers."""
    def __init__(self, width, height, lambd, gamma, reward_rb, reward_rf, cost_rb, min_start_wealth, p, beta, alpha, num_neighbors=20):
        super().__init__()
        # self.num_agents = N
        self.width = width
        self.height = height
        self.grid = SingleGrid(width, height, torus=True)

        self.lambd = lambd
        self.gamma = gamma
        self.reward_rb = reward_rb
        self.reward_rf = reward_rf
        self.cost_rb = cost_rb
        self.p = p
        self.beta = beta
        self.alpha = alpha

        self.num_neighbors = num_neighbors
        self.desperate_state = 0  # Initialize desperate state to 0 (not desperate)
        self.decision = 0  # Initialize decision to 0 (follow rules)
        self.caught = False  # Initialize caught state to False
        self.income_rank = 0  # Initialize income rank to 0
        self.rb_choice = 0  # Initialize rule-breaking choice to 0 (follow rules)

        # Create agents and place them on the grid
        for _, pos in self.grid.coord_iter():
            # Create a new agent at the current position
            agent = individual(
                model=self, lambd=self.lambd, gamma=self.gamma, reward_rb=self.reward_rb,
                reward_rf=self.reward_rf, cost_rb=self.cost_rb, min_start_wealth=min_start_wealth,
                p=self.p, beta=self.beta, alpha=self.alpha
            )
            # Add the agent to the grid at the current position
            self.grid.place_agent(agent, pos)

    
        self.datacollector = mesa.DataCollector(
            model_reporters={"Proportion crime": crime_proportion},
            agent_reporters={ "Wealth Start": lambda a: getattr(a, "wealth_start", a.wealth), "Wealth End": lambda a: getattr(a, "wealth_end", a.wealth), "income rank":"income_rank", "desperate_state": "desperate_state", 
                             "Rule-breaking choice": "decision", "Caught": "caught", "SV_rule_break": "SV_rule_break", "SV_follow_rules": "SV_follow_rules"}
            )
        self.running = True
        self.datacollector.collect(self)
    
    def step(self):
        # self.datacollector.collect(self)
        self.agents.do("step")
        self.datacollector.collect(self)
        

    

        


In [36]:
# Run the model for a number of steps
test_model = rel_DMAP_model(lambd=1, gamma=0.3, reward_rb=500, reward_rf=100, cost_rb=3000, min_start_wealth=100, p=0.1, beta=1.8, alpha=0.8, width=5, height=5)
for _ in range(50):
    test_model.step()

model_df = test_model.datacollector.get_model_vars_dataframe()
agent_df = test_model.datacollector.get_agent_vars_dataframe()

Agent 1 is not relatively desperate with income rank 0.30434782608695654
Agent 1 chose to break the rules with probability 0.9988179536829973
Agent 2 is not relatively desperate with income rank 0.391304347826087
Agent 2 chose to break the rules with probability 0.9985490458297629
Agent 3 is not relatively desperate with income rank 0.9130434782608695
Agent 3 chose to break the rules with probability 0.980747818628109
Agent 4 is not relatively desperate with income rank 0.17391304347826086
Agent 4 chose to break the rules with probability 0.9988639021937208
Agent 5 is not relatively desperate with income rank 0.5217391304347826
Agent 5 chose to break the rules with probability 0.9972046798068852
Agent 6 is not relatively desperate with income rank 0.08695652173913043
Agent 6 chose to break the rules with probability 0.9988734511687029
Agent 7 is not relatively desperate with income rank 0.21739130434782608
Agent 7 chose to break the rules with probability 0.9987100502804195
Agent 8 is 

In [37]:
agent_df.head(51)

Wealth Start   Wealth End  income rank  desperate_state  \
Step AgentID                                                            
0    1           15.120797    15.120797     0.000000                0   
     2           33.388709    33.388709     0.000000                0   
     3          422.568258   422.568258     0.000000                0   
     4           11.783694    11.783694     0.000000                0   
     5          103.715918   103.715918     0.000000                0   
     6           11.081372    11.081372     0.000000                0   
     7           22.696821    22.696821     0.000000                0   
     8           23.157383    23.157383     0.000000                0   
     9          222.652111   222.652111     0.000000                0   
     10         117.320882   117.320882     0.000000                0   
     11          99.021507    99.021507     0.000000                0   
     12        1334.001305  1334.001305     0.000000                0   
     13           7.754413     7.754413     0.000000                0   
     14         305.657154   305.657154     0.000000                0   
     15          13.912215    13.912215     0.000000                0   
     16         263.210171   263.210171     0.000000                0   
     17          14.205879    14.205879     0.000000                0   
     18          11.301096    11.301096     0.000000                0   
     19         131.416571   131.416571     0.000000                0   
     20          68.568608    68.568608     0.000000                0   
     21          59.968266    59.968266     0.000000                0   
     22          47.143120    47.143120     0.000000                0   
     23         123.005447   123.005447     0.000000                0   
     24           6.651039     6.651039     0.000000                0   
     25           4.984132     4.984132     0.000000                0   
1    1           15.120797   415.120797     0.304348                0   
     2           33.388709   433.388709     0.391304                0   
     3          422.568258   822.568258     0.913043                0   
     4           11.783694   411.783694     0.173913                0   
     5          103.715918   503.715918     0.521739                0   
     6           11.081372   411.081372     0.086957                0   
     7           22.696821   422.696821     0.217391                0   
     8           23.157383   423.157383     0.217391                0   
     9          222.652111 -2877.347889     0.521739                0   
     10         117.320882   517.320882     0.434783                0   
     11          99.021507   499.021507     0.391304                0   
     12        1334.001305 -1765.998695     1.000000                0   
     13           7.754413   407.754413     0.130435                0   
     14         305.657154   705.657154     0.521739                0   
     15          13.912215   413.912215     0.173913                0   
     16         263.210171   663.210171     0.434783                0   
     17          14.205879   414.205879     0.173913                0   
     18          11.301096   411.301096     0.130435                0   
     19         131.416571   531.416571     0.304348                0   
     20          68.568608   468.568608     0.217391                0   
     21          59.968266   459.968266     0.173913                0   
     22          47.143120   447.143120     0.130435                0   
     23         123.005447   523.005447     0.130435                0   
     24           6.651039   406.651039     0.086957                0   
     25           4.984132   404.984132     0.043478                0   
2    1          415.120797   815.120797     0.391304                0   

              Rule-breaking choice  Caught  SV_rule_break  SV_follow_rules  
Step AgentID                                                       

In [31]:
# (ambiguity aversion)/beta mean: 0.12; median: 0.09, SD: 0.41, min: -0.97 max: 0.97
# (a-insensitivity)/alpha mean: 0.41 median: 0.29, SD: 0.44 min: -0.22 max: 2.2
parameters = {"gamma": 0.3,
              "reward_rb": [500, 1000],
              "reward_rf": [50, 100],
              "cost_rb": [500, 1000],
              "min_start_wealth": [200,1000],
              "p": 0.1,
              "beta": [-0.97, 0.97],
              "alpha": [-0.22, 2.2],
              "lambd": 1,  # Add lambda parameter
              "width": 5,  # Add width parameter
              "height": 5}  # Add height parameter
results = mesa.batch_run(
    rel_DMAP_model,
    parameters,
    iterations=1,
    max_steps=10,
    data_collection_period=1,
    number_processes=1  # Change to use multiple CPU cores for parallel execution
)

  0%|          | 0/64 [00:00<?, ?it/s]

Agent 1 is not relatively desperate with income rank 0.391304347826087
Agent 1 chose to break the rules with probability [0.98883087]
Agent 2 is not relatively desperate with income rank 0.5652173913043478
Agent 2 chose to break the rules with probability [0.99472419]
Agent 3 is not relatively desperate with income rank 0.043478260869565216
Agent 3 chose to break the rules with probability [0.98219912]
Agent 4 is not relatively desperate with income rank 0.13043478260869565
Agent 4 chose to break the rules with probability [0.98601926]
Agent 5 is not relatively desperate with income rank 0.2608695652173913
Agent 5 chose to break the rules with probability [0.98834778]
Agent 6 is not relatively desperate with income rank 0.43478260869565216
Agent 6 chose to break the rules with probability [0.99649271]
Agent 7 is not relatively desperate with income rank 0.9130434782608695
Agent 7 chose to break the rules with probability [1.]
Agent 8 is not relatively desperate with income rank 0.17391

In [32]:
results_df = pd.DataFrame(results)

results_df.head(51)

,RunId,iteration,Step,gamma,reward_rb,reward_rf,cost_rb,min_start_wealth,p,beta,...,Proportion crime,AgentID,Wealth Start,Wealth End,income rank,desperate_state,Rule-breaking choice,Caught,SV_rule_break,SV_follow_rules
0,0,0,0,0.3,500,50,500,200,0.1,-0.97,...,0.00,1,[2670.9502381801353],[2670.9502381801353],0.000000,0,0,False,None,None
1,0,0,0,0.3,500,50,500,200,0.1,-0.97,...,0.00,2,[-158.38294602012093],[-158.38294602012093],0.000000,0,0,False,None,None
2,0,0,0,0.3,500,50,500,200,0.1,-0.97,...,0.00,3,[3616.2089807520074],[3616.2089807520074],0.000000,0,0,False,None,None
3,0,0,0,0.3,500,50,500,200,0.1,-0.97,...,0.00,4,[4645.84690641646],[4645.84690641646],0.000000,0,0,False,None,None
4,0,0,0,0.3,500,50,500,200,0.1,-0.97,...,0.00,5,[3666.3761786131454],[3666.3761786131454],0.000000,0,0,False,None,None
5,0,0,0,0.3,500,50,500,200,0.1,-0.97,...,0.00,6,[4772.980735185642],[4772.980735185642],0.000000,0,0,False,None,None
6,0,0,0,0.3,500,50,500,200,0.1,-0.97,...,0.00,7,[5545.422373994852],[5545.422373994852],0.000000,0,0,False,None,None
7,0,0,0,0.3,500,50,500,200,0.1,-0.97,...,0.00,8,[2650.513882250262],[2650.513882250262],0.000000,0,0,False,None,None
8,0,0,0,0.3,500,50,500,200,0.1,-0.97,...,0.00,9,[1353.8908446242376],[1353.8908446242376],0.000000,0,0,False,None,None
9,0,0,0,0.3,500,50,500,200,0.1,-0.97,...,0.00,10,[4723.5447652739485],[4723.5447652739485],0.000000,0,0,False,None,None


In [181]:
# Save the results to a CSV file
results_df.to_csv("relative_desp_results.csv", index=False)

In [ ]:
# Get rid of the rows where "Step" is 0
results_df = results_df[results_df["Step"] > 0]


## Data Visualization

In [ ]:
# Calculate the Gini coefficient for the population
def gini_coefficient(wealths):
    """Calculate the Gini coefficient of a list of wealth values."""
    n = len(wealths)
    if n == 0:
        return 0
    sorted_wealths = np.sort(wealths)
    cumulative_wealths = np.cumsum(sorted_wealths)
    total_wealth = cumulative_wealths[-1]
    if total_wealth == 0:
        return 0
    gini_index = (2 * np.sum((np.arange(1, n + 1) * sorted_wealths))) / (n * total_wealth) - (n + 1) / n
    return gini_index
gini = gini_coefficient(results_df["Wealth End"])
print(f"Gini Coefficient: {gini}")


In [ ]:
# Show distribution of wealth at the end of the simulation
plt.figure(figsize=(10, 6))
sns.histplot(results_df["Wealth End"], bins=30, kde=True)
plt.title("Distribution of Wealth at the End of Simulation")
plt.xlabel("Wealth End")
plt.ylabel("Frequency")
plt.show()


In [ ]:
# Show relationship between wealth and rule-breaking behavior
plt.figure(figsize=(10, 6))
sns.scatterplot(x="Wealth End", y="Rule-breaking choice", data=results_df, alpha=0.5)
plt.title("Wealth vs. Rule-breaking Behavior")      


In [ ]:
# Show the relationship between income rank and rule-breaking behavior
plt.figure(figsize=(10, 6))
sns.scatterplot(x="income rank", y="Rule-breaking choice", data=results_df, alpha=0.5)
plt.title("Income Rank vs. Rule-breaking Behavior")
plt.xlabel("Income Rank")
plt.ylabel("Rule-breaking choice")

In [ ]:
# Show the relationship between income rank and the expected utility of rule-breaking and following rules
plt.figure(figsize=(10, 6))
sns.scatterplot(x="income rank", y="SV_rule_break", data=results_df, alpha=0.5, label="Expected Utility of Rule Breaking")
sns.scatterplot(x="income rank", y="SV_follow_rules", data=results_df, alpha=0.5, label="Expected Utility of Following Rules")
plt.title("Income Rank vs. Expected Utility of Rule-breaking and Following Rules")
plt.xlabel("Income Rank")
plt.ylabel("Expected Utility")
plt.legend()
plt.show()

### Grid Visualization